[Use PyTorch for all questions]

[Disclaimer: This part of the course has been updated, it is the first time I teach it (please be gentle)]

In [ ]:
import time
from pathlib import Path
from urllib.request import urlretrieve
from zipfile import ZipFile

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
device

## Classifying heartbeats with a 1D CNN

Our goal: from the recording of **one heartbeat** (an electrocardiogram, ECG), predict whether it is normal, or whether the patient had a myocardial infarction (a heart attack).

In Session 9, we used `nn.Conv2d` on images: a small kernel slides over the two spatial dimensions.
A signal is an "image" with a single dimension, time: `nn.Conv1d` slides a small kernel along time, and detects local patterns (a peak, a steep slope, a notch, ...) wherever they occur.

The whole practical runs on a laptop CPU: the dataset is tiny (200 heartbeats), and so are the networks.

We will:
1. write our own PyTorch `Dataset`, reading the original data files;
2. train dense baselines;
3. understand `nn.Conv1d`, then train a 1D CNN, and compare it with a recurrent network;
4. see what happens when heartbeats are not perfectly aligned, and use our `Dataset` for data augmentation;
5. look at what the CNN learned.

### 0. Data

**ECG200** (Olszewski, 2001), from the [UCR/UEA time series classification archive](https://www.timeseriesclassification.com/): 200 heartbeats of 96 time steps each, already split into 100 training and 100 test heartbeats.

In [ ]:
data_dir = Path("../data/ECG200")
if not data_dir.exists():
    urlretrieve("https://www.timeseriesclassification.com/aeon-toolkit/ECG200.zip", "ECG200.zip")
    ZipFile("ECG200.zip").extractall(data_dir)
    Path("ECG200.zip").unlink()
sorted(path.name for path in data_dir.iterdir())

The archive contains the same data in three formats (`.arff`, `.ts` and `.txt`), and a description of the dataset.
We will use the `.txt` files.

Print the description (`ECG200.txt`), and the beginning (say the first 500 characters) of `ECG200_TRAIN.txt`.
What is the format of the file? What is in the first column?

*Your answer here.*

### 1. A custom `Dataset`

PyTorch's `DataLoader` builds the batches from a `Dataset`: any object with two methods,
- `__len__()`: the number of examples,
- `__getitem__(i)`: the `i`-th example (here, a pair `(signal, label)`).

For each batch, the `DataLoader` calls `dataset[i]` for a few indices `i`, and stacks the results into tensors.

In Session 9, `torchvision.datasets.ImageFolder` was a ready-made `Dataset`, for images sorted in one folder per class.
There is no ready-made `Dataset` for our files, so we write our own (subclass of `torch.utils.data.Dataset`).

Implement a class `ECGDataset(path, transform=None)`:
- `__init__` reads the file (`np.loadtxt`) and stores:
    - `self.signals`: a float tensor of shape `[n_signals, length]`,
    - `self.classes`: the sorted list of the labels found in the file (here `[-1, 1]`),
    - `self.labels`: a tensor of integers in $\{0, \dots, \text{n\_classes} - 1\}$ (the index of each label in `self.classes`), as expected by `F.cross_entropy`;
- `__len__` returns the number of signals;
- `__getitem__(i)` returns the pair `(signal, label)` of the `i`-th heartbeat, where `signal` has shape `[1, length]`; if `transform` is not `None`, it is applied to the signal before returning it (as the `transform` of `ImageFolder` in Session 9: we will use it in part 4).

*Why `[1, length]`?* `nn.Conv1d` expects batches of shape `[batch, channels, length]`. Here we have a single channel (one electrode); a recording with 12 electrodes would have 12 channels, like an RGB image has 3.

In [ ]:
d = ECGDataset(data_dir / "ECG200_TRAIN.txt")
d.signals.shape, d.classes

Create `train_dataset` and `test_dataset` from `ECG200_TRAIN.txt` and `ECG200_TEST.txt`. Check:
- their sizes and their classes,
- the shape and type of the signal and the label of one example,
- the mean and standard deviation of a few signals: were they normalized?

*Your answer here.*

Create a `DataLoader` for each dataset (batches of 16, shuffle the training set only), and check the shapes of one batch.

Label $1$ is a normal heartbeat, label $-1$ a myocardial infarction:

In [ ]:
names = {-1: "infarction", 1: "normal"}
class_names = [names[c] for c in train_dataset.classes]
class_names

Plot a few heartbeats of each class (one subplot per class). Do you see a difference between the two classes?

Then, compute the proportion of each class in the training and test sets.
What accuracy does a model get if it always predicts the most frequent class?

*Your answer here.*

### 2. Dense baselines

Write a function `train_model(model, train_loader, epochs=100, lr=1e-3, weight=None)` that trains `model` with Adam and the cross-entropy loss (`weight` is passed to `F.cross_entropy`, we will use it in the bonus), and returns the list of the average training losses of each epoch.

Write a function `accuracy(model, loader)` that returns the accuracy of `model` on the data of `loader`.

With only 100 test heartbeats, one heartbeat is 1% of accuracy, and two trainings of the same model (with a different random initialization) easily differ by a few %.
To compare models fairly, we train each of them several times (with different seeds) and report the mean and the standard deviation of the test accuracy.

In [ ]:
def n_params(model):
    return sum(p.numel() for p in model.parameters())


def train_seeds(make_model, train_loader=train_loader, n_seeds=5, epochs=100):
    """Train make_model() n_seeds times; return the trained models and their losses."""
    models, losses = [], []
    start = time.time()
    for seed in range(n_seeds):
        torch.manual_seed(seed)
        model = make_model()
        losses.append(train_model(model, train_loader, epochs=epochs))
        models.append(model)
    print(f"{n_params(model):,} parameters, trained {n_seeds} times in {time.time() - start:.0f}s")
    return models, losses


def summary(models, loader, seed=1):
    torch.manual_seed(seed)  # (useful in part 4: all models see the same random shifts)
    accuracies = [accuracy(model, loader) for model in models]
    return f"{np.mean(accuracies):.1%} ± {np.std(accuracies):.1%}"

Train (with `train_seeds`) and evaluate (with `summary`):
- a logistic regression: a single `nn.Linear` layer, on the 96 values of the (flattened) signal,
- an MLP with one hidden layer of 64 neurons.

Store the models in a dictionary `models` (we will reuse them), and plot the training losses.

In [ ]:
models, losses = {}, {}

models["logistic regression"], losses["logistic regression"] = train_seeds(
    lambda: nn.Sequential(nn.Flatten(), nn.Linear(96, 2))
)

In [ ]:
models["MLP"], losses["MLP"] = train_seeds(
    lambda: nn.Sequential(nn.Flatten(), nn.Linear(96, 64), nn.ReLU(), nn.Linear(64, 2))
)

In [ ]:
for name in models:
    print(f"{name:20} train {summary(models[name], train_loader)}, test {summary(models[name], test_loader)}")

for name in losses:
    plt.semilogy(np.mean(losses[name], axis=0), label=name)
plt.xlabel("epoch")
plt.ylabel("training loss")
plt.legend()
plt.show()

*Your answer here.*

### 3. 1D convolutions

#### 3.a. One convolution

Let's first look at a single `nn.Conv1d(in_channels=1, out_channels=1, kernel_size=5)`.
Apply it to a batch made of the first training heartbeat (shape `[1, 1, 96]`).
- What is the shape of the output? Why?
- Recompute the output "by hand", from `conv.weight` and `conv.bias` (with a loop, or with `signal.unfold`), and check that you get the same values.
- What is the length of the output with `padding=2`? With `padding="same"`? With `stride=2`?

*Your answer here.*

The kernel decides what the convolution detects.
Set by hand the weights of a `nn.Conv1d(1, 1, kernel_size=3, padding=1)` so that it computes the (discrete) derivative of the signal: $y_t = \frac{x_{t+1} - x_{t-1}}{2}$.
Plot the first heartbeat and its derivative, then the output of the derivative followed by a ReLU. What does the ReLU keep?

*Your answer here.*

#### 3.b. A 1D CNN

Implement a class `ECGConvNet(kernel_size=7, n_classes=2)`:
- 3 convolutional blocks, with 16, 32 and 32 channels. Each block is a `nn.Conv1d` (with `padding="same"`), followed by a ReLU, and by a `nn.MaxPool1d(2)` for the first two blocks;
- a **global average pooling** (`nn.AdaptiveAvgPool1d(1)`): each channel is averaged over time;
- a `nn.Linear` layer to the `n_classes` scores.

Print the shape of the output of each layer on a batch.

In [ ]:



model = ECGConvNet()
x = train_dataset[0][0].unsqueeze(0)
print("input:", tuple(x.shape))
for layer in [model.conv1,model.maxpool1,model.conv2,model.maxpool2,model.conv3, model.pool, nn.Flatten(), model.out]:
    x = layer(x)
    print(f"{layer.__class__.__name__:18}", tuple(x.shape))

Why a global average pooling, instead of flattening the output of the last convolution (as we did in Session 9)?

*Your answer here.*

Train it (with `train_seeds`), evaluate it, and compare it with the dense baselines (number of parameters, accuracy).

In [ ]:
models["CNN (k=7)"], losses["CNN (k=7)"] = train_seeds(lambda: ECGConvNet(kernel_size=7))

In [ ]:
for name in models:
    print(f"{name:20} train {summary(models[name], train_loader)}, test {summary(models[name], test_loader)}")

*Your answer here.*

#### 3.c. Kernel size

The **receptive field** of a neuron is the part of the input it depends on.
A neuron after the first convolution sees $k$ time steps; after the max pooling, $k + 1$ time steps; etc.

What is the receptive field of a neuron of the last convolution of `ECGConvNet`, as a function of the kernel size $k$? Compute it for $k = 3$, $7$ and $15$, and compare it to the length of a heartbeat.

*Your answer here.*

Train and evaluate the CNN with kernels of size 3 and 15. Which one is the best? Why?

In [ ]:
models["CNN (k=3)"], losses["CNN (k=3)"] = train_seeds(lambda: ECGConvNet(kernel_size=3))

In [ ]:
models["CNN (k=15)"], losses["CNN (k=15)"] = train_seeds(lambda: ECGConvNet(kernel_size=15))

In [ ]:
for name in models:
    print(f"{name:20} train {summary(models[name], train_loader)}, test {summary(models[name], test_loader)}")

*Your answer here.*

#### 3.d. And a recurrent network?

In Session 10, we read sequences with recurrent networks. A heartbeat is a sequence of 96 values, so a GRU can read it one time step at a time, and summarize it in its hidden state.

Implement a class `ECGRecurrentNet(hidden_size=32, pool="last", n_classes=2)`:
- a `nn.GRU(input_size=1, hidden_size=hidden_size, batch_first=True)`. Careful with the shapes: `nn.GRU` expects `[batch, length, features]`, while our signals are `[batch, channels, length]` (what `nn.Conv1d` expects): transpose them,
- the heartbeat is summarized by the hidden state at the last time step (`pool="last"`), or by the average of the hidden states over all time steps (`pool="mean"`),
- a `nn.Linear(hidden_size, n_classes)`.

Train and evaluate both versions, and compare them with the previous models (number of parameters, accuracy, training time).

In [ ]:
models["GRU (last)"], losses["GRU (last)"] = train_seeds(lambda: ECGRecurrentNet(pool="last"))

In [ ]:
models["GRU (mean)"], losses["GRU (mean)"] = train_seeds(lambda: ECGRecurrentNet(pool="mean"))

In [ ]:
for name in models:
    print(f"{name:20} train {summary(models[name], train_loader)}, test {summary(models[name], test_loader)}")

*Your answer here.*

### 4. Misaligned heartbeats

In ECG200, every heartbeat was cut around its R peak, so that all heartbeats are aligned.
In a real recording, the heartbeats are cut automatically, and the window can start a bit earlier or a bit later.

Let's simulate this, by **shifting** each heartbeat by a random number of time steps (between $-10$ and $10$) with `torch.roll`.
This is a **transformation** of the signal, like the random flips of the images in Session 9: write it as a class `RandomShift(max_shift=10)` whose instances can be called on a signal (method `__call__(signal)`), and give it to the dataset with `ECGDataset(path, transform=RandomShift())`.
The transformation is applied in `__getitem__`, so the shift is drawn each time an example is read: each read gives a new shift.

Create a shifted test set (and its `DataLoader`), and plot a heartbeat as read 3 times from it.

Then, evaluate the models trained above on the shifted test set (`summary` fixes the seed, so that all models see the same shifts).
Which models suffer the most from the shifts? Why?

In [ ]:
shifted_test_dataset = ECGDataset(data_dir / "ECG200_TEST.txt", transform=RandomShift())
shifted_test_loader = DataLoader(shifted_test_dataset, batch_size=16)

for _ in range(3):
    plt.plot(shifted_test_dataset[0][0][0])
plt.title("the same heartbeat, read 3 times")
plt.xlabel("time step")
plt.show()

for name in models:
    print(f"{name:20} test {summary(models[name], test_loader)}, shifted test {summary(models[name], shifted_test_loader)}")

*Your answer here.*

**Data augmentation:** if the test heartbeats can be shifted, we can show shifted heartbeats during training, so that the model learns not to care about shifts.
Since `RandomShift` draws a new shift every time an example is read, every epoch sees a new version of each heartbeat: our 100 heartbeats become a (virtually) infinite dataset.

Train the MLP, the CNN ($k = 15$) and the two GRUs on a shifted training set (one cell per model), and evaluate them on the test set and on the shifted test set. Conclude.

In [ ]:
shifted_train_dataset = ECGDataset(data_dir / "ECG200_TRAIN.txt", transform=RandomShift())
shifted_train_loader = DataLoader(shifted_train_dataset, batch_size=16, shuffle=True)

In [ ]:
models["MLP + shifts"], losses["MLP + shifts"] = train_seeds(
    lambda: nn.Sequential(nn.Flatten(), nn.Linear(96, 64), nn.ReLU(), nn.Linear(64, 2)),
    train_loader=shifted_train_loader,
)

In [ ]:
models["CNN (k=15) + shifts"], losses["CNN (k=15) + shifts"] = train_seeds(
    lambda: ECGConvNet(kernel_size=15), train_loader=shifted_train_loader
)

In [ ]:
models["GRU (last) + shifts"], losses["GRU (last) + shifts"] = train_seeds(
    lambda: ECGRecurrentNet(pool="last"), train_loader=shifted_train_loader
)

In [ ]:
models["GRU (mean) + shifts"], losses["GRU (mean) + shifts"] = train_seeds(
    lambda: ECGRecurrentNet(pool="mean"), train_loader=shifted_train_loader
)

In [ ]:
for name in models:
    print(f"{name:20} test {summary(models[name], test_loader)}, shifted test {summary(models[name], shifted_test_loader)}")

*Your answer here.*

### 5. What did the CNN learn?

Take the first CNN ($k = 15$) trained with shifts. Plot:
- the 16 kernels of its first convolution,
- for one normal and one infarction heartbeat of the test set, the 16 feature maps after the first convolution and its ReLU.

Can you interpret some of the kernels?

In [ ]:
model = models["CNN (k=15) + shifts"][0].cpu().eval()
kernels = model.conv1.weight.detach()[:, 0]  # [16, 15]

fig, axes = plt.subplots(2, 8, figsize=(16, 4), sharey=True)
for kernel, ax in zip(kernels, axes.flat):
    ax.plot(kernel, ".-")
    ax.axhline(0, color="gray", lw=0.5)
plt.suptitle("kernels of the first convolution")
plt.show()

examples = [(test_dataset.labels == c).nonzero()[0].item() for c in range(2)]
fig, axes = plt.subplots(2, 2, figsize=(16, 7), sharex=True, height_ratios=[1, 2])
for i, (ax_signal, ax_maps) in zip(examples, axes.T):
    signal, label = test_dataset[i]
    with torch.no_grad():
        maps = model.conv1(signal.unsqueeze(0))[0].clamp(0, None)  # [16, 96]
    ax_signal.plot(signal[0])
    ax_signal.set_title(f"{class_names[label]} heartbeat")
    ax_maps.imshow(maps, aspect="auto", cmap="viridis", interpolation="nearest")
    ax_maps.set_xlabel("time step")
    ax_maps.set_ylabel("channel")
plt.show()

*Your answer here.*

-----

### 6. Bonus: ECG5000

Nothing in `ECGDataset` is specific to ECG200: it reads any dataset of the archive in the same format.

**ECG5000** contains 5000 heartbeats of 140 time steps, in 5 classes: class 1 is a normal heartbeat, classes 2 to 5 are different kinds of abnormal heartbeats. The archive provides only 500 heartbeats for training, and 4500 for testing.

Download it (`https://www.timeseriesclassification.com/aeon-toolkit/ECG5000.zip`), create the datasets and the dataloaders, and look at the class proportions.

Train an `ECGConvNet` (only `n_classes` changes: thanks to the global pooling, the length of the signals does not matter) for 50 epochs.
Compute its test accuracy and its **recall** on each class (the proportion of the heartbeats of the class that are correctly classified). What do you observe?

*Your answer here.*

Train it again, with a **weighted** loss: the weight of each class is inversely proportional to its number of training heartbeats (`weight` argument of `train_model`). Compare.

*Your answer here.*